In [1]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [5]:
# =============================================
# AUTOMATED FINANCIAL PERFORMANCE TRACKER
# =============================================

import yfinance as yf
import pandas as pd
import numpy as np  

# User-defined list of stock ticker symbols (must match Yahoo Finance format)
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

# Analysis period: 3 years of historical data
PERIOD = "3y"

# Risk-free rate for Sharpe Ratio calculation (annualized, as a decimal)
RISK_FREE_RATE = 0.03  # Represents a 3% annual risk-free rate


# ------------------------------------------------------------------------------
# FUNCTION: fetch_stock_data()
# Purpose : Downloads adjusted closing prices for all tickers in one API call.
# Returns : A Pandas DataFrame where each column is one ticker's daily price.
# ------------------------------------------------------------------------------

def fetch_stock_data(tickers: list, period: str) -> pd.DataFrame:
    """
    Fetches historical adjusted closing prices for a list of stock tickers.

    Args:
        tickers (list): A list of ticker symbols, e.g., ["AAPL", "MSFT"].
        period (str) : The look-back period, e.g., "3y" for 3 years.

    Returns:
        pd.DataFrame: A DataFrame of daily adjusted closing prices.
                      Rows = dates, Columns = ticker symbols.
    """

    print(f"Fetching data for: {tickers}...")

    # yf.download() fetches ALL tickers in a single, efficient bulk API call.
    # 'auto_adjust=True' uses the adjusted close price, which accounts for
    # stock splits and dividends — this is the correct price for return analysis.
    raw_data = yf.download(
        tickers=tickers,
        period=period,
        auto_adjust=True,
        progress=False  # Suppresses the yfinance download progress bar
    )

    # yf.download() returns a multi-level column DataFrame.
    # We only need the 'Close' prices (one column per ticker).
    price_data = raw_data["Close"]

    # Data Quality Check: Warn if any ticker has excessive missing data.
    missing_pct = price_data.isnull().mean() * 100
    for ticker, pct in missing_pct.items():
        if pct > 5:
            print(f"  WARNING: '{ticker}' has {pct:.1f}% missing data. "
                  f"Check if this is a valid ticker symbol.")

    # Drop any rows where ALL tickers have missing data (e.g., market holidays)
    price_data = price_data.dropna(how="all")

    print(f"  Success! Retrieved {len(price_data)} trading days of data.\n")

    return price_data

# ===================
# Metrics Engine
# ===================

def calculate_metrics(price_data: pd.DataFrame, risk_free_rate: float) -> pd.DataFrame:
    """
    Calculates 3-Year CAGR, Annualized Volatility, and Sharpe Ratio
    for each ticker in the price DataFrame.

    Args:
        price_data (pd.DataFrame) : Daily adjusted closing prices (from Step 1).
        risk_free_rate (float)    : Annual risk-free rate as a decimal (e.g., 0.03).

    Returns:
        pd.DataFrame: A summary table with one row per ticker and three metric columns.
    """

    # Calculate Daily Log Returns
    # Log return formula: ln(P_today / P_yesterday)
    # We use .pct_change() to get simple returns first, then convert.
    # Why log returns? They are symmetric and additive across time periods —
    # the mathematically correct choice for multi-period return analysis.
    log_returns = np.log(1 + price_data.pct_change()).dropna()

    # We'll store each ticker's results as a dictionary, then build a DataFrame.
    results = []

    for ticker in price_data.columns:

        # Drop any NaN values for this specific ticker (handles IPO or delisting gaps)
        ticker_prices  = price_data[ticker].dropna()
        ticker_returns = log_returns[ticker].dropna()

        # Calculate 3-Year CAGR 
        # CAGR formula: (Ending Value / Beginning Value) ^ (1 / Years) - 1
        # This gives the "smoothed" annual growth rate that would produce
        # the same ending value as the actual path taken.
        start_price = ticker_prices.iloc[0]   # First available price
        end_price   = ticker_prices.iloc[-1]  # Last available price

        # Calculate exact number of years from the actual date range
        n_years = len(ticker_prices) / 252    # 252 = average trading days per year

        cagr = (end_price / start_price) ** (1 / n_years) - 1

        # Calculate Annualized Volatility 
        # Daily volatility = standard deviation of daily log returns
        # Annualized by multiplying by √252 (square root of trading days per year)
        # This scaling rule comes from the property of variance being additive:
        # Annual Variance = Daily Variance × 252, so Annual Std Dev = Daily Std Dev × √252
        daily_vol      = ticker_returns.std()
        annual_vol     = daily_vol * np.sqrt(252)

        # Calculate Sharpe Ratio 
        # Sharpe Ratio = (Portfolio Return - Risk-Free Rate) / Portfolio Volatility
        # This tells us: "How much excess return are we earning per unit of risk?"
        # A Sharpe > 1.0 is generally considered good; > 2.0 is excellent.
        sharpe_ratio = (cagr - risk_free_rate) / annual_vol

        # Append this ticker's results to our list
        results.append({
            "Ticker"               : ticker,
            "3-Year CAGR (%)"      : round(cagr * 100, 2),       # Convert to percentage
            "Annual Volatility (%)" : round(annual_vol * 100, 2), # Convert to percentage
            "Sharpe Ratio"         : round(sharpe_ratio, 2)
        })

    # Convert the list of dictionaries into a clean DataFrame
    metrics_df = pd.DataFrame(results).set_index("Ticker")

    return metrics_df

# ========================================
#  Master Function + Excel Export
# ========================================

import os                          # For handling file paths
from datetime import datetime      # For timestamping the output file

def export_to_excel(metrics_df: pd.DataFrame, prices_df: pd.DataFrame, output_path: str):
    """
    Exports the metrics summary and raw price data to a professionally
    formatted Excel workbook with two sheets.

    Args:
        metrics_df  (pd.DataFrame): The summary metrics table (from Step 2).
        prices_df   (pd.DataFrame): The raw daily price data (from Step 1).
        output_path (str)         : Full file path for the output .xlsx file.
    """

    # pd.ExcelWriter lets us write multiple sheets to one workbook.
    # engine='openpyxl' is required for formatting — make sure it's installed.
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

        #  Sheet 1: Executive Summary (the metrics table) 
        metrics_df.to_excel(writer, sheet_name='Performance Summary', startrow=3)

        # Access the underlying openpyxl workbook objects so we can format them
        workbook  = writer.book
        ws_summary = writer.sheets['Performance Summary']

        # Define reusable styles using openpyxl's PatternFill and Font
        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        # Header fill color: dark navy blue (standard for analyst reports)
        header_fill  = PatternFill("solid", fgColor="1F3864")
        header_font  = Font(color="1F3864", bold=True, name="Calibri", size=11)
        title_font   = Font(bold=True, name="Calibri", size=14, color="1F3864")
        subtext_font = Font(name="Calibri", size=10, color="7F7F7F")

        thin_border_side = Side(style='thin', color="D9D9D9")
        row_border = Border(
            bottom=Border(bottom=thin_border_side).bottom
        )

        # Write the report title in Row 1 and generation date in Row 2
        ws_summary['A1'] = "Financial Performance Tracker — Executive Summary"
        ws_summary['A1'].font = title_font

        ws_summary['A2'] = f"Generated: {datetime.now().strftime('%B %d, %Y')}"
        ws_summary['A2'].font = subtext_font

        # Style the header row (Row 4, because startrow=3 means headers land on row 4)
        header_row = 4
        for col_num in range(1, len(metrics_df.columns) + 2):  # +2 for index column
            cell = ws_summary.cell(row=header_row, column=col_num)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center')

        # Style the data rows with alternating light grey fill for readability
        light_fill = PatternFill("solid", fgColor="F2F2F2")
        for row_num in range(header_row + 1, header_row + 1 + len(metrics_df)):
            for col_num in range(1, len(metrics_df.columns) + 2):
                cell = ws_summary.cell(row=row_num, column=col_num)
                # Apply zebra striping: every other row gets a light grey fill
                if (row_num - header_row) % 2 == 0:
                    cell.fill = light_fill
                cell.alignment = Alignment(horizontal='center')

        # Auto-fit column widths so nothing gets cut off
        for col_num in range(1, len(metrics_df.columns) + 2):
            col_letter = get_column_letter(col_num)
            ws_summary.column_dimensions[col_letter].width = 22

        # Sheet 2: Raw Price Data (for reference and further analysis)
        prices_df.round(2).to_excel(writer, sheet_name='Historical Prices', startrow=1)
        ws_prices = writer.sheets['Historical Prices']

        ws_prices['A1'] = "Daily Adjusted Closing Prices (USD)"
        ws_prices['A1'].font = title_font

        # Style the header row on Sheet 2 as well
        for col_num in range(1, len(prices_df.columns) + 2):
            cell = ws_prices.cell(row=2, column=col_num)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center')
            ws_prices.column_dimensions[get_column_letter(col_num)].width = 16

    print(f"  Excel report saved to: {output_path}")


# ==========================================================
# THE MASTER FUNCTION — this is the one the user calls
# ==========================================================

def run_tracker(tickers: list, period: str = "3y", risk_free_rate: float = 0.03):
    """
    Master pipeline function. Fetches data, calculates metrics, exports to
    Excel, and generates a performance chart — all in a single call.

    Args:
        tickers        (list) : List of ticker symbols, e.g. ["AAPL", "MSFT"].
        period         (str)  : Look-back period (default: "3y").
        risk_free_rate (float): Annual risk-free rate as decimal (default: 0.03).

    Returns:
        pd.DataFrame: The metrics summary table (also saved as .xlsx and .png).
    """

    print("=" * 55)
    print("   AUTOMATED FINANCIAL PERFORMANCE TRACKER")
    print("=" * 55)

    #  Fetch
    prices = fetch_stock_data(tickers=tickers, period=period)

    #  Calculate
    print("Calculating performance metrics...")
    metrics = calculate_metrics(price_data=prices, risk_free_rate=risk_free_rate)
    print(metrics.to_string())
    print()

    #  Export to Excel
    print("Exporting to Excel...")
    timestamp   = datetime.now().strftime("%Y%m%d_%H%M")
    output_dir  = "output"
    os.makedirs(output_dir, exist_ok=True)   # Creates /output folder if it doesn't exist
    excel_path  = os.path.join(output_dir, f"performance_report_{timestamp}.xlsx")
    export_to_excel(metrics_df=metrics, prices_df=prices, output_path=excel_path)

   #  Visualize
    print("Generating performance chart...")
    chart_path = os.path.join(output_dir, f"cumulative_returns_{timestamp}.png")
    plot_cumulative_returns(prices_df=prices, output_path=chart_path)
    
    print("\nAll done! Check the /output folder for your report.")
    print("=" * 55)

    return metrics

# =================================================
#  Visualization — Cumulative Returns Chart
# =================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

def plot_cumulative_returns(prices_df: pd.DataFrame, output_path: str):
    """
    Plots the cumulative return of each ticker over the analysis period
    and saves the chart as a high-resolution PNG file.

    Args:
        prices_df   (pd.DataFrame): Daily adjusted closing prices (from Step 1).
        output_path (str)         : Full file path for the output .png file.
    """

    # Calculate cumulative returns
    # Step 1: Get daily % change for each ticker
    # Step 2: Add 1 so a 2% day becomes 1.02 (a multiplier, not a percentage)
    # Step 3: .cumprod() multiplies these multipliers together across all days
    
    cumulative_returns = (1 + prices_df.pct_change()).cumprod()

    # Set the first row to exactly 1.0 (the "base" starting point)
    cumulative_returns.iloc[0] = 1.0

    # Build the chart
    # Use seaborn's clean "whitegrid" style for a professional look
    sns.set_style("whitegrid")

    fig, ax = plt.subplots(figsize=(12, 6))  # Wide format suits financial charts

    # Plot each ticker as a separate line
    for ticker in cumulative_returns.columns:
        ax.plot(
            cumulative_returns.index,
            cumulative_returns[ticker],
            label=ticker,
            linewidth=2
        )

    # --- Formatting ---
    # Convert y-axis to percentage labels (1.35 → "+35%")
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f"{(x - 1) * 100:+.0f}%"
    ))

    # Add a horizontal reference line at 0% (break-even)
    ax.axhline(y=1.0, color='black', linewidth=0.8, linestyle='--', alpha=0.4,
               label='Break-even (0%)')

    # Titles and labels
    ax.set_title(
        "3-Year Cumulative Return Comparison",
        fontsize=16, fontweight='bold', pad=15
    )
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Cumulative Return", fontsize=12)
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)

    # Rotate date labels so they don't overlap
    plt.xticks(rotation=30, ha='right')

    # Tight layout prevents labels from being cut off at the edges
    plt.tight_layout()

    # Save as high-resolution PNG (300 DPI is print-quality)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()  # Close the figure to free memory — good practice in scripts

    print(f"  Chart saved to: {output_path}")

# ------------------------------------------------------------------------------
# MAIN EXECUTION BLOCK
# This block only runs when you execute this script directly (not when imported).
# It's a Python best practice for keeping code modular and reusable.
# ------------------------------------------------------------------------------

if __name__ == "__main__":
    results = run_tracker(tickers=TICKERS, period=PERIOD, risk_free_rate=RISK_FREE_RATE)

   AUTOMATED FINANCIAL PERFORMANCE TRACKER
Fetching data for: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']...
  Success! Retrieved 753 trading days of data.

Calculating performance metrics...
        3-Year CAGR (%)  Annual Volatility (%)  Sharpe Ratio
Ticker                                                      
AAPL              20.34                  25.73          0.67
AMZN              24.90                  31.00          0.71
GOOGL             43.11                  29.67          1.35
MSFT               8.56                  24.03          0.23
TSLA              20.98                  57.46          0.31

Exporting to Excel...
  Excel report saved to: output/performance_report_20260607_1804.xlsx
Generating performance chart...
  Chart saved to: output/cumulative_returns_20260607_1804.png

All done! Check the /output folder for your report.


In [4]:
run_tracker(
    tickers=["RELIANCE.NS", "TCS.NS", "INFY.NS"],
    period="3y",
    risk_free_rate=0.067  # Use Indian 10-yr bond yield as risk-free rate for Indian stocks
)

   AUTOMATED FINANCIAL PERFORMANCE TRACKER
Fetching data for: ['RELIANCE.NS', 'TCS.NS', 'INFY.NS']...
  Success! Retrieved 743 trading days of data.

Calculating performance metrics...
             3-Year CAGR (%)  Annual Volatility (%)  Sharpe Ratio
Ticker                                                           
INFY.NS                -0.42                  25.19         -0.28
RELIANCE.NS             4.76                  20.87         -0.09
TCS.NS                 -9.84                  22.16         -0.75

Exporting to Excel...
  Excel report saved to: output/performance_report_20260607_1727.xlsx
Generating performance chart...
  Chart saved to: output/cumulative_returns_20260607_1727.png

All done! Check the /output folder for your report.


,3-Year CAGR (%),Annual Volatility (%),Sharpe Ratio
Ticker,,,
INFY.NS,-0.42,25.19,-0.28
RELIANCE.NS,4.76,20.87,-0.09
TCS.NS,-9.84,22.16,-0.75
